### CASE TECNICO PYSPARK

In [1]:
##Import list
import pandas as pd
import matplotlib as plt
import os
import warnings
import logging
from pyspark.sql import SparkSession

# Suppress non-critical warnings
warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)

#set up global variables

CWD = os.getcwd()
CLIENTES_PATH = os.path.join(CWD, 'data/clients/data.json')
PEDIDOS_PATH = os.path.join(CWD, 'data/pedidos/data.json')

#start pyspark session

spark = (
    SparkSession.builder
    .appName("CaseTecnico")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

# Set log level AFTER session creation
spark.sparkContext.setLogLevel("ERROR")



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 11:01:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, StringType, StructType, StructField
from pyspark.storagelevel import StorageLevel
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql.functions import broadcast

# explicit schemas avoid extra pass for inference
CLIENTES_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
])

PEDIDOS_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("value", DecimalType(5, 2), True),
])


def load_data_from_json(path: str, schema: StructType, min_partitions: int | None = None) -> DataFrame:
    """Fast JSONL loader: schema-first, lazy, and optional repartition. Persists DataFrame in memory."""
    df = (
        spark.read
        .schema(schema)
        .option("multiLine", "false")
        .option("mode", "PERMISSIVE")
        .json(path)
    )

    if min_partitions is not None and df.rdd.getNumPartitions() < min_partitions:
        df = df.repartition(min_partitions)

    df = df.persist(StorageLevel.MEMORY_AND_DISK)
    return df



clientes_df = load_data_from_json(CLIENTES_PATH, CLIENTES_SCHEMA)
pedidos_df= load_data_from_json(PEDIDOS_PATH, PEDIDOS_SCHEMA, min_partitions=32)

print("clientes partitions:", clientes_df.rdd.getNumPartitions())
print("pedidos partitions:", pedidos_df.rdd.getNumPartitions())
print("schemas loaded successfully")

clientes partitions: 1


pedidos partitions: 32
schemas loaded successfully


## 1. Data Quality - Relatório de Falhas

In [55]:
def regra_falha(df, motivo: str, ordem: int):
    return df.select(
        F.col("id"),
        F.lit(motivo).alias("motivo"),
        F.lit(ordem).alias("ordem_regra")
    )

# 2) ID de pedido duplicado
ids_duplicados_df = (
    pedidos_df
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .select("id")
)
ids_duplicados = regra_falha(ids_duplicados_df, "id_duplicado", 1)

# 1) Pedido sem valor
Sem_valor = regra_falha(
    pedidos_df.filter(F.col("value").isNull()),
    "pedido_sem_valor",
    2
)


# 3) Pedido com cliente inexistente (somente client_id válido)
cliente_inexistente = (
    pedidos_df.alias("p")
    .filter(F.col("p.client_id").isNotNull() & (F.col("p.client_id") < 0))
    .join(
        broadcast(clientes_df.select(F.col("id").alias("client_id_ref")).alias("c")),
        F.col("p.client_id") == F.col("c.client_id_ref"),
        "left_anti"
    )
    .select(F.col("p.id").alias("id"))
    .transform(lambda df: regra_falha(df, "cliente_inexistente", 3))
)

# 4) ID nulo
id_nullo = regra_falha(
    pedidos_df.filter(F.col("id").isNull()),
    "id_nulo",
    4
)

# 5) client_id nulo
client_id_nulo = regra_falha(
    pedidos_df.filter(F.col("client_id").isNull()),
    "client_id_nulo",
    5   
)

# 6) ID inválido (< 0)
id_invalido = regra_falha(
    pedidos_df.filter(F.col("id").isNotNull() & (F.col("id") < 0)),
    "id_invalido_menor_igual_zero",
    6
)

# 7) client_id inválido (<= 0)
client_id_invalido = regra_falha(
    pedidos_df.filter(F.col("client_id").isNotNull() & (F.col("client_id") < 0)),
    "client_id_invalido_menor_igual_zero",
    7
)

# 8) valor zero
valor_zero = regra_falha(
    pedidos_df.filter(F.col("value") == 0),
    "pedido_com_valor_zero",
    8
)

# 9) Retorno inválido - verifica se pedidos com o mesmo ID têm valor positivo suficiente
# Regra: se um ID tem valor negativo, deve existir valor positivo no mesmo ID >= |valor_negativo|
# Agrupa por ID e calcula total positivo e negativo
pedidos_por_id_df = (
    pedidos_df
    .filter(F.col("value").isNotNull())
    .groupBy("id")
    .agg(
        F.sum(F.when(F.col("value") > 0, F.col("value")).otherwise(0)).alias("total_positivo"),
        F.abs(F.sum(F.when(F.col("value") < 0, F.col("value")).otherwise(0))).alias("total_negativo_abs")
    )
)

# IDs com valores negativos mas sem valor positivo suficiente para cobrir
retornos_invalidos_df = (
    pedidos_por_id_df
    .filter(
        (F.col("total_negativo_abs") > 0) &  # Tem valor negativo
        (F.col("total_positivo") < F.col("total_negativo_abs"))  # Positivo não cobre o negativo
    )
    .select("id")
)

retornos_invalidos = regra_falha(retornos_invalidos_df, "retorno_sem_cobertura_positiva", 9)

# União de todas as regras
regras_falhas = [
    Sem_valor, ids_duplicados, cliente_inexistente, id_nullo, client_id_nulo,
    id_invalido, client_id_invalido, valor_zero, retornos_invalidos
]

falhas_df = (
    reduce(lambda acc, d: acc.unionByName(d), regras_falhas)
    .dropDuplicates(["id", "motivo"])
    .groupBy("id")
    .agg(
        F.concat_ws(", ", F.array_sort(F.collect_list("motivo"))).alias("motivo"),
        F.min("ordem_regra").alias("min_ordem")
    )
    .orderBy("min_ordem", "id")
    .select("id", "motivo")
)

# Saídas pedidas
falhas_df.show(100, truncate=False)

erros_por_categoria_df = (
    falhas_df
    .groupBy("motivo")
    .agg(F.count("*").alias("qtd_erros"))
    .orderBy(F.col("qtd_erros").desc(), F.col("motivo").asc())
)

erros_por_categoria_df.show(truncate=False)
falhas_df.agg(F.count("*").alias("total_erros")).show()


+------+--------------------------------------------------------------+
|id    |motivo                                                        |
+------+--------------------------------------------------------------+
|1534  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva|
|3443  |id_duplicado                                                  |
|3502  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva|
|3679  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva|
|4322  |id_duplicado, pedido_sem_valor                                |
|4724  |id_duplicado, pedido_sem_valor                                |
|5774  |id_duplicado, pedido_sem_valor                                |
|6322  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva|
|6622  |id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva|
|6884  |id_duplicado, pedido_sem_valor                                |
|7116  |id_duplicado, pedido_sem_valor                          

+--------------------------------------------------------------+---------+
|motivo                                                        |qtd_erros|
+--------------------------------------------------------------+---------+
|id_duplicado, pedido_sem_valor                                |25178    |
|id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva|24807    |
|id_duplicado                                                  |4506     |
+--------------------------------------------------------------+---------+

+-----------+
|total_erros|
+-----------+
|      54491|
+-----------+



In [56]:
# Step 1: Filter to rows with valid values first
pedidos_with_valid_values = (
    pedidos_df
    .select("id", "client_id", "value")
    .filter(
        F.col("value").isNotNull() 
        & (F.col("value") > 0)
        & F.col("id").isNotNull() 
        & (F.col("id") >= 0)
        & F.col("client_id").isNotNull() 
        & (F.col("client_id") >= 0)
    )
)

# Step 2: Check for duplicates ONLY among valid values
# (if ID appears multiple times but only once with valid value, we keep it)
ids_duplicados_validos_df = (
    pedidos_with_valid_values
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Prepare client IDs as DataFrame (not Python list)
clientes_ids_df = clientes_df.select(F.col("id").alias("client_id_ref")).distinct()

# Step 3: Keep only non-duplicated valid orders with existing clients
pedidos_validos_df = (
    pedidos_with_valid_values
    # Anti-join to exclude IDs that are duplicated among valid values
    .join(broadcast(ids_duplicados_validos_df), F.col("id") == F.col("dup_id"), "left_anti")
    # Inner join to validate client exists
    .join(broadcast(clientes_ids_df), F.col("client_id") == F.col("client_id_ref"), "inner")
    .select("id", "client_id", "value")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pedidos_validos = pedidos_validos_df.count()
total_pedidos = pedidos_df.count()

print("Total pedidos:", total_pedidos)
print("Pedidos válidos:", pedidos_validos)

Total pedidos: 1100000
Pedidos válidos: 989978


In [58]:
# Optimized: aggregate first (reduces data size), then join with broadcast
cliente_totals_df = (
    pedidos_validos_df
    .groupBy("client_id")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.sum("value").alias("valor_total"),
    )
    .join(
        broadcast(clientes_df.select(
            F.col("id").alias("client_id_ref"), 
            F.col("name").alias("client_name")
        )),
        F.col("client_id") == F.col("client_id_ref"),
        "inner"
    )
    .select(
        F.col("client_id").alias("id_cliente"),
        F.col("client_name").alias("nome_cliente"),
        F.col("qtd_pedidos"),
        F.col("valor_total")
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

cliente_totals_df.show(50, truncate=False)

+----------+----------------+-----------+-----------+
|id_cliente|nome_cliente    |qtd_pedidos|valor_total|
+----------+----------------+-----------+-----------+
|123456    |Inês Siqueira   |494418     |24946507.01|
|9047      |Zachary Reis    |64         |4213.80    |
|4494      |Vitor Marques   |72         |4185.48    |
|2756      |Inês Siqueira   |65         |3977.05    |
|8317      |Sofia Castro    |72         |3954.30    |
|8566      |Tereza Leal     |72         |3940.74    |
|2379      |Gustavo Pontes  |70         |3932.49    |
|849       |Breno Soares    |72         |3932.45    |
|7543      |Vitória Andrade |75         |3877.94    |
|1704      |Eduardo Santos  |60         |3866.91    |
|5221      |Yasmin Carvalho |69         |3855.91    |
|9147      |Zachary Reis    |63         |3836.79    |
|1942      |Ulisses Moraes  |68         |3836.04    |
|9266      |Tereza Leal     |70         |3826.47    |
|2695      |Wanda Silva     |63         |3824.19    |
|5602      |Carlos Souza    

In [59]:
# Optimized: compute statistics in minimal passes (2 actions instead of 3)
stats = cliente_totals_df.agg(F.mean("valor_total").alias("media")).collect()[0]
media = stats["media"]

# Compute all quantiles in a single pass
percentil_10, mediana, percentil_90 = cliente_totals_df.approxQuantile("valor_total", [0.1, 0.5, 0.9], 0.01)

print(f"Valor médio total por cliente: {media:.2f}")
print(f"Mediana do valor total por cliente: {mediana:.2f}")
print(f"10º percentil do valor total por cliente: {percentil_10:.2f}")
print(f"90º percentil do valor total por cliente: {percentil_90:.2f}")

Valor médio total por cliente: 4996.23
Mediana do valor total por cliente: 2490.95
10º percentil do valor total por cliente: 1985.99
90º percentil do valor total por cliente: 3033.04


In [60]:
clientes_acima_media_df = (
    cliente_totals_df
    .filter(F.col("valor_total") > media)
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df.show(50, truncate=False)

+----------+-------------+-----------+-----------+
|id_cliente|nome_cliente |qtd_pedidos|valor_total|
+----------+-------------+-----------+-----------+
|123456    |Inês Siqueira|494418     |24946507.01|
+----------+-------------+-----------+-----------+



In [61]:
clientes_media_truncada_df = (
    cliente_totals_df
    .filter(
        (F.col("valor_total") >= percentil_10) &
        (F.col("valor_total") <= percentil_90)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

+----------+-----------------+-----------+-----------+
|id_cliente|nome_cliente     |qtd_pedidos|valor_total|
+----------+-----------------+-----------+-----------+
|7801      |Bruno Costa      |58         |3033.04    |
|1600      |Ana Silva        |62         |3032.76    |
|5767      |Ubirajara Sampaio|58         |3032.74    |
|4883      |Klaus Magalhães  |62         |3032.38    |
|114       |Otávio Cardoso   |60         |3032.34    |
|5546      |Yara Macedo      |48         |3032.20    |
|9818      |Thiago Araújo    |57         |3032.11    |
|2523      |Amanda Rocha     |56         |3032.10    |
|2321      |Yasmin Carvalho  |55         |3031.97    |
|3101      |Bruno Costa      |53         |3031.84    |
|7304      |Eduardo Santos   |58         |3031.72    |
|4507      |Helena Rodrigues |59         |3031.33    |
|9927      |Elisa Campos     |55         |3031.22    |
|187       |Oscar Vasconcelos|61         |3031.03    |
|8347      |Zachary Reis     |58         |3030.30    |
|5154     

## Detecção e Remoção de Outliers - Cliente ID 123456

### Justificativa Técnica para Remoção:

**1. Anomalia Estatística Extrema:**
- Cliente 123456: **494,418 pedidos** | Valor total: R$ 24,946,507
- Outros clientes: ~60-75 pedidos | Valor total: R$ 3,000-4,200
- **Razão: 6,500x mais pedidos que a média**

**2. Distribuição de Valores Suspeita:**
- 79 pedidos com valor 99.72 (idênticos)
- 77 pedidos com valor 59.25 (idênticos)
- 76 pedidos com valor 51.41 (idênticos)
- ~70+ pedidos com valores praticamente idênticos (diferença de poucos centavos)
- **Padrão de repetição muito improvável em dados reais**

**3. Impacto nos Cálculos Estatísticos:**
- **Com cliente 123456:**
  - Média: R$ 4,996.23 (inflacionada em ~2x)
  - Mediana: R$ 2,490.95
  - 90º percentil: R$ 3,033.04
  
- **Sem cliente 123456:**
  - Média: R$ 2,502.08 (normalizada)
  - Mediana: R$ 2,486.66 (praticamente sem alteração)
  - 90º percentil: R$ 3,020.61

**4. Conclusão:**
Este cliente é claramente um caso de **erro de dados** (possível duplicação, teste não removido ou corrupção). Deve ser excluído de análises estatísticas por violar pressupostos de qualidade de dados.

In [ ]:
pedidos_ines = (
    pedidos_validos_df.alias("pedidos")
    .filter(F.col("client_id") == 123456)
    .orderBy(F.col("value").desc())
    .select("id", "value")
)

pedidos_ines.show(10,truncate=False)

quantidade_ines99 = pedidos_ines.filter(F.col("value") == 99.99).count()

print("Quantidade de pedidos de Inês com valor 99.99:", quantidade_ines99)

Ines_valores_repitidos_df = (
    pedidos_ines.groupBy("value")
    .agg(F.count("*").alias("qtd_repeticoes"))
    .filter(F.col("qtd_repeticoes") > 1)
    .orderBy(F.col("qtd_repeticoes").desc(), F.col("value").asc())
)

Ines_valores_repitidos_df.show(10, truncate=False)

+--------+-----+
|id      |value|
+--------+-----+
|68789339|99.99|
|50816623|99.99|
|94693379|99.99|
|1230531 |99.99|
|29038115|99.99|
|73735842|99.99|
|63255621|99.99|
|64450996|99.99|
|95922104|99.99|
|33658689|99.99|
+--------+-----+
only showing top 10 rows
Quantidade de pedidos de Inês com valor 99.99: 27
+-----+--------------+
|value|qtd_repeticoes|
+-----+--------------+
|99.72|79            |
|59.25|77            |
|51.41|76            |
|75.98|75            |
|58.09|74            |
|71.72|74            |
|83.85|74            |
|84.41|74            |
|6.86 |73            |
|31.87|73            |
+-----+--------------+
only showing top 10 rows


In [ ]:
# Remove outlier client 123456 and recalculate with clean data
pedidos_validos_clean_df = (
    pedidos_validos_df
    .filter(F.col("client_id") != 123456)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pedidos_validos_clean = pedidos_validos_clean_df.count()

print("=" * 60)
print("ANÁLISE COM DADOS LIMPOS (sem cliente outlier 123456)")
print("=" * 60)
print(f"Pedidos válidos (original): {pedidos_validos}")
print(f"Pedidos válidos (sem 123456): {pedidos_validos_clean}")
print(f"Pedidos removidos: {pedidos_validos - pedidos_validos_clean}")
print("=" * 60)

In [63]:
cliente_totals_df.filter(F.col("id_cliente") != 123456).show(truncate=False)   

+----------+---------------+-----------+-----------+
|id_cliente|nome_cliente   |qtd_pedidos|valor_total|
+----------+---------------+-----------+-----------+
|9047      |Zachary Reis   |64         |4213.80    |
|4494      |Vitor Marques  |72         |4185.48    |
|2756      |Inês Siqueira  |65         |3977.05    |
|8317      |Sofia Castro   |72         |3954.30    |
|8566      |Tereza Leal    |72         |3940.74    |
|2379      |Gustavo Pontes |70         |3932.49    |
|849       |Breno Soares   |72         |3932.45    |
|7543      |Vitória Andrade|75         |3877.94    |
|1704      |Eduardo Santos |60         |3866.91    |
|5221      |Yasmin Carvalho|69         |3855.91    |
|9147      |Zachary Reis   |63         |3836.79    |
|1942      |Ulisses Moraes |68         |3836.04    |
|9266      |Tereza Leal    |70         |3826.47    |
|2695      |Wanda Silva    |63         |3824.19    |
|5602      |Carlos Souza   |74         |3818.54    |
|6135      |Mariana Melo   |74         |3811.2

In [64]:
media_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).agg(F.mean("valor_total").alias("media_valor_total")).collect()[0]["media_valor_total"]

print(f"Valor médio total por cliente: {media_A:.2f}")

mediana_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).approxQuantile("valor_total", [0.5], 0.01)[0]

print(f"Mediana do valor total por cliente: {mediana_A:.2f}")

percentil_10_A, percentil_90_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).approxQuantile("valor_total", [0.1, 0.9], 0.01)

print(f"10º percentil do valor total por cliente: {percentil_10_A:.2f}")

print(f"90º percentil do valor total por cliente: {percentil_90_A:.2f}")

Valor médio total por cliente: 2502.08
Mediana do valor total por cliente: 2486.66
10º percentil do valor total por cliente: 1979.81
90º percentil do valor total por cliente: 3020.61


In [65]:
clientes_acima_media_df_A = (
    cliente_totals_df
    .filter(
        (F.col("id_cliente") != 123456) & (F.col("valor_total") > media_A)
        )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df_A.show(50, truncate=False)

+----------+----------------+-----------+-----------+
|id_cliente|nome_cliente    |qtd_pedidos|valor_total|
+----------+----------------+-----------+-----------+
|9047      |Zachary Reis    |64         |4213.80    |
|4494      |Vitor Marques   |72         |4185.48    |
|2756      |Inês Siqueira   |65         |3977.05    |
|8317      |Sofia Castro    |72         |3954.30    |
|8566      |Tereza Leal     |72         |3940.74    |
|2379      |Gustavo Pontes  |70         |3932.49    |
|849       |Breno Soares    |72         |3932.45    |
|7543      |Vitória Andrade |75         |3877.94    |
|1704      |Eduardo Santos  |60         |3866.91    |
|5221      |Yasmin Carvalho |69         |3855.91    |
|9147      |Zachary Reis    |63         |3836.79    |
|1942      |Ulisses Moraes  |68         |3836.04    |
|9266      |Tereza Leal     |70         |3826.47    |
|2695      |Wanda Silva     |63         |3824.19    |
|5602      |Carlos Souza    |74         |3818.54    |
|6135      |Mariana Melo    

In [66]:
clientes_media_truncada_df_A = (
    cliente_totals_df
    .filter(
        (F.col("id_cliente") != 123456) &
        (F.col("valor_total") >= percentil_10_A) &
        (F.col("valor_total") <= percentil_90_A)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

+----------+-----------------+-----------+-----------+
|id_cliente|nome_cliente     |qtd_pedidos|valor_total|
+----------+-----------------+-----------+-----------+
|7801      |Bruno Costa      |58         |3033.04    |
|1600      |Ana Silva        |62         |3032.76    |
|5767      |Ubirajara Sampaio|58         |3032.74    |
|4883      |Klaus Magalhães  |62         |3032.38    |
|114       |Otávio Cardoso   |60         |3032.34    |
|5546      |Yara Macedo      |48         |3032.20    |
|9818      |Thiago Araújo    |57         |3032.11    |
|2523      |Amanda Rocha     |56         |3032.10    |
|2321      |Yasmin Carvalho  |55         |3031.97    |
|3101      |Bruno Costa      |53         |3031.84    |
|7304      |Eduardo Santos   |58         |3031.72    |
|4507      |Helena Rodrigues |59         |3031.33    |
|9927      |Elisa Campos     |55         |3031.22    |
|187       |Oscar Vasconcelos|61         |3031.03    |
|8347      |Zachary Reis     |58         |3030.30    |
|5154     